In [2]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import os



# ====== INISIALISASI OPTIONS ======
edge_options = Options()
edge_options.add_argument(r"user-data-dir=D:\Tools\selenium_edge_profile")
edge_options.add_argument("profile-directory=Stockbit")
edge_options.add_argument('--disable-blink-features=AutomationControlled')
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option('useAutomationExtension', False)

Path_Webdriver = "./Driver/msedgedriver.exe"
target_url = {
    "Barang-Konsumen-Primer": {
        "Perdagangan Ritel Barang Primer": "https://stockbit.com/sector/barang-konsumen-primer/perdagangan-ritel-barang-primer",
        "Makanan-Minuman": "https://stockbit.com/sector/barang-konsumen-primer/makanan-minuman",
        "Produk-Rumah-Tangga-Tidak-Tahan-Lama": "https://stockbit.com/sector/barang-konsumen-primer/produk-rumah-tangga-tidak-tahan-lama",
        "Rokok": "https://stockbit.com/sector/barang-konsumen-primer/rokok"
    },
    "Keuangan": {
        "Asuransi": "https://stockbit.com/sector/keuangan/asuransi",
        "jasa-pembiayaan": "https://stockbit.com/sector/keuangan/jasa-pembiayaan",
        "Bank": "https://stockbit.com/sector/keuangan/bank",
        "jasa-investasi": "https://stockbit.com/sector/keuangan/jasa-investasi",
        "holding-investasi": "https://stockbit.com/sector/keuangan/perusahaan-holding-investasi"
    },
    "Barang-Konsumen-NonPrimer": {
        "media-hiburan": "https://stockbit.com/sector/barang-konsumen-non-primer/media-hiburan",
        "perdagangan-ritel": "https://stockbit.com/sector/barang-konsumen-non-primer/perdagangan-ritel",
        "pakaian-barang-mewah": "https://stockbit.com/sector/barang-konsumen-non-primer/pakaian-barang-mewah",
        "Komponen-Otomotif": "https://stockbit.com/sector/barang-konsumen-non-primer/otomotif-komponen-otomotif",
        "Barang-Rekreasi": "https://stockbit.com/sector/barang-konsumen-non-primer/barang-rekreasi",
        "Barang-Rumah-Tangga": "https://stockbit.com/sector/barang-konsumen-non-primer/barang-rumah-tangga",
        "Jasa-Konsumen": "https://stockbit.com/sector/barang-konsumen-non-primer/Jasa-konsumen"
    },
    "Energi": {
        "Minyak-Gas-BatuBara": "https://stockbit.com/sector/energi/minyak-gas-batu-bara",
        "Energi-Alternatif": "https://stockbit.com/sector/energi/energi-alternatif"
    },
    "Teknologi": {
        "Hardware": "https://stockbit.com/sector/teknologi/perangkat-keras-peralatan-teknologi"
    },
    "Kesehatan": {
        "Farmasi-Riset": "https://stockbit.com/sector/kesehatan/farmasi-riset-kesehatan"
    },
    "Properti": {
        "Properti-RealEstate": "https://stockbit.com/sector/properti-real-estat/properti-dan-real-estate"
    },
    "Barang-Baku": {
        "Barang-Baku": "https://stockbit.com/sector/barang-baku/barang-baku"
    },
    "Perindustrian": {
        "Perusahaan-Holding": "https://stockbit.com/sector/Perindustrian/perusahaan-holding-multi-sektor",
        "Barang-Perindustrian": "https://stockbit.com/sector/Perindustrian/barang-perindustrian"
    },
    "Transportasi": {
        "Transportasi": "https://stockbit.com/sector/transportasi-logistik/transportasi",
        "Logistik": "https://stockbit.com/sector/transportasi-logistik/logistik-pengantaran"
    }
}

output = "List-Perusahaan/Perusahaan.csv"
os.makedirs(os.path.dirname(output), exist_ok=True)

# ====== INISIALISASI WEBDRIVER ======
service = Service(executable_path=Path_Webdriver)
driver = webdriver.Edge(options=edge_options)
wait = WebDriverWait(driver, 10)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

driver.get("https://stockbit.com/login")
time.sleep(3)
try:
    login_button = wait.until(EC.presence_of_element_located((By.ID, "google-login-button")))
    login_button.click()
except Exception:
    print("Sepertinya sudah login")
time.sleep(2)

# ============================================================
# Fungsi: Scroll via scrollIntoView pada elemen terakhir
# Cara paling reliable untuk virtual scroll — browser yang urus sendiri
# ============================================================
def scrape_with_virtual_scroll(driver):
    """
    Kumpulkan ticker dengan scroll otomatis sampai tidak ada item baru.
    Trick: scrollIntoView pada elemen TERAKHIR → container scroll ikut.
    """
    collected = {}
    no_new_streak = 0
    MAX_NO_NEW = 3  # Berhenti setelah 3x scroll berturut-turut tanpa item baru

    while True:
        # Ambil semua link yang saat ini ada di DOM
        all_links = driver.find_elements(
            By.CSS_SELECTOR, 'div.ant-table-container tbody a[href^="/symbol/"]'
        )

        count_before = len(collected)

        for el in all_links:
            ticker = el.get_attribute("textContent").strip()
            if not ticker:
                ticker = (el.get_attribute("title") or "").strip() or el.text.strip()
            if ticker and ticker not in collected:
                link = el.get_attribute("href") or ''
                collected[ticker] = link

        new_items = len(collected) - count_before

        if new_items > 0:
            no_new_streak = 0
        else:
            no_new_streak += 1
            if no_new_streak >= MAX_NO_NEW:
                break  # Tidak ada item baru setelah beberapa kali scroll → selesai

        # ✅ KEY FIX: scrollIntoView pada elemen TERAKHIR di DOM
        # Ini paksa container virtual scroll untuk render baris berikutnya
        if all_links:
            try:
                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'nearest', behavior: 'smooth'});",
                    all_links[-1]
                )
            except Exception:
                pass

        time.sleep(0.6)  # Tunggu render virtual rows baru

    return collected

# ============================================================
# Fungsi: Klik Next Page dengan scroll ke tombol dulu
# ============================================================
def click_next_page(driver):
    try:
        next_btn = driver.find_element(
            By.CSS_SELECTOR, "li.ant-pagination-next:not(.ant-pagination-disabled)"
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
        time.sleep(0.4)
        next_btn.click()
        return True
    except Exception:
        return False

# ============================================================
# Main loop
# ============================================================
all_data = []

for sektor, sub_sektor_dict in target_url.items():
    print(f"\n{'='*60}")
    print(f"Processing Sektor: {sektor}")
    print(f"{'='*60}")

    for sub_sektor, url in sub_sektor_dict.items():
        print(f"\n  → Sub-sektor: {sub_sektor}")

        try:
            driver.get(url)
            wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, 'div.ant-table-container tbody a[href^="/symbol/"]')
                )
            )
            print(f"    ✓ Tabel muncul")
            time.sleep(1)

            seen_tickers = set()
            sub_sektor_count = 0
            page_num = 1

            while True:
                # Scrape satu halaman penuh (termasuk virtual scroll)
                page_tickers = scrape_with_virtual_scroll(driver)

                new_this_page = 0
                for ticker, link in page_tickers.items():
                    if ticker not in seen_tickers:
                        seen_tickers.add(ticker)
                        all_data.append({
                            "Sektor": sektor,
                            "Sub_Sektor": sub_sektor,
                            "Ticker": ticker,
                            "Ticker_YF": f"{ticker}.JK",
                            "Link": link
                        })
                        sub_sektor_count += 1
                        new_this_page += 1

                print(f"    → Halaman {page_num}: {new_this_page} ticker (total DOM terlihat: {len(page_tickers)})")

                # Pindah ke halaman berikutnya (pagination)
                has_next = click_next_page(driver)
                if not has_next:
                    break

                page_num += 1
                time.sleep(2)
                wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, 'div.ant-table-container tbody a[href^="/symbol/"]')
                    )
                )
                time.sleep(0.5)

            print(f"    ✓ TOTAL: {sub_sektor_count} perusahaan | Akumulasi: {len(all_data)}")

        except Exception as e:
            print(f"    ✗ Error di {sub_sektor}: {str(e)}")
            continue



# Simpan
df = pd.DataFrame(all_data)
df.to_csv(output, index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print(f"✓ Tersimpan ke: {output}")
print(f"✓ Total perusahaan: {len(df)}")
print(f"{'='*60}")
print("\nPreview:")
print(df.head(10))
print(f"\nPer sektor:")
print(df['Sektor'].value_counts())

driver.quit()


Sepertinya sudah login

Processing Sektor: Barang-Konsumen-Primer

  → Sub-sektor: Perdagangan Ritel Barang Primer
    ✓ Tabel muncul
    → Halaman 1: 19 ticker (total DOM terlihat: 19)
    ✓ TOTAL: 19 perusahaan | Akumulasi: 19

  → Sub-sektor: Makanan-Minuman
    ✓ Tabel muncul
    → Halaman 1: 100 ticker (total DOM terlihat: 100)
    ✓ TOTAL: 100 perusahaan | Akumulasi: 119

  → Sub-sektor: Produk-Rumah-Tangga-Tidak-Tahan-Lama
    ✓ Tabel muncul
    → Halaman 1: 13 ticker (total DOM terlihat: 13)
    ✓ TOTAL: 13 perusahaan | Akumulasi: 132

  → Sub-sektor: Rokok
    ✓ Tabel muncul
    → Halaman 1: 5 ticker (total DOM terlihat: 5)
    ✓ TOTAL: 5 perusahaan | Akumulasi: 137

Processing Sektor: Keuangan

  → Sub-sektor: Asuransi
    ✓ Tabel muncul
    → Halaman 1: 19 ticker (total DOM terlihat: 19)
    ✓ TOTAL: 19 perusahaan | Akumulasi: 156

  → Sub-sektor: jasa-pembiayaan
    ✓ Tabel muncul
    → Halaman 1: 13 ticker (total DOM terlihat: 13)
    ✓ TOTAL: 13 perusahaan | Akumulasi: 16